# Week 2: Check and Revise an AI Response

Your support workflow now receives the complaint, invoice findings, and company rules. You want it to check its answer before returning advice. Implement the review loop and test whether it corrects mistakes or introduces them.

Read the lab and rubric in Canvas. Keep the six support tasks from Week 1. Generate one answer per sample, then compare no review, self-review, and review with external feedback. Both review paths start from the same saved answer.

The file and folder names below are defaults for the supplied helpers. Your submission can keep the same evidence in notebook cells, separate files, or a combination, under names you choose.

## 1. Inspect the inputs

Run the load cell from your COSC-651-Labs checkout. In Colab, its file picker also accepts a ZIP of that repository. It loads the supplied code and public data without installing a course package. Keep your work before disconnecting; runtime files are separate from the saved notebook.

The missing-date preview shows the exact request, diagnostic, and current policies. Sample names describe the inputs; internal identifiers are retained for traceability.

In [ ]:
import io, json, os, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
lab_folder = next((p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / 'src/maintenance_course/review_lab.py').is_file()), None)
if lab_folder is None:
    from google.colab import files
    print('Select a ZIP of the COSC-651 Labs repository.')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    assert len(archives) == 1, 'Upload exactly one student package ZIP.'
    lab_folder = Path('/content/cosc-651-labs')
    assert not lab_folder.exists(), 'Keep existing work; use a new runtime for a different package.'
    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        names = set(archive.namelist())
        projects = [n for n in names if PurePosixPath(n).name == 'pyproject.toml'
                    and n[:-len('pyproject.toml')] + 'src/maintenance_course/review_lab.py' in names
                    and n[:-len('pyproject.toml')] + 'week-02/week-02.ipynb' in names]
        assert len(projects) == 1, 'The ZIP must contain one COSC-651 Labs repository.'
        prefix = projects[0][:-len('pyproject.toml')]
        allowed = {'src', 'data', 'tests', 'student_work', 'README.md', 'AGENTS.md', 'pyproject.toml'}
        allowed.update(f'week-{number:02d}' for number in range(1, 9))
        members = [m for m in archive.infolist() if m.filename.startswith(prefix) and not m.is_dir()
                   and PurePosixPath(m.filename[len(prefix):]).parts[0] in allowed]
        assert sum(m.file_size for m in members) < 200_000_000, 'Unexpected lab archive size.'
        for member in members:
            target = (lab_folder / Path(*PurePosixPath(member.filename[len(prefix):]).parts)).resolve()
            assert target.is_relative_to(lab_folder.resolve()), 'Invalid archive path.'
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
sys.path[:0] = [str(lab_folder / 'src'), str(lab_folder)]
from maintenance_course.data import LAB_ROOT
from maintenance_course.review_lab import (ReviewFixture, prepare_task, development_tasks, run_comparison, resource_rows, rating_form, export_comparison)
sys.path.insert(0, str(LAB_ROOT))
ARTIFACT_ROOT = Path(os.getenv('COURSE_REVIEW_ARTIFACTS', str(LAB_ROOT / 'artifacts/week02')))
task = prepare_task('missing-date')
print(task['prompt'])


## 2. Define success before running

Judge whether each answer supports its factual claims, follows applicable rules, and completes the six support tasks. A justified request for missing information can succeed. Overall success also requires the three code checks and no execution error.

Write your retention rule before enabling live requests. Include the defects you want to reduce, regressions you would reject, and additional work you would accept. Keep the rule unchanged for this comparison group.

In [ ]:
retention_rule = ''  # Your predeclared criteria.
EXPERIMENT = 'comparison-01'  # A new name for changed inputs, settings, route, or a repaired experiment.
MAX_ROUNDS = 2
RUN_STUDENT = False
RUN_CONTROL_TESTS = False
RUN_LIVE_PILOT = False
RUN_LIVE_BATCH = False
EXPORT = False
DOWNLOAD = False
assert EXPERIMENT and all(c.isalnum() or c in '-_' for c in EXPERIMENT)
assert type(MAX_ROUNDS) is int and 1 <= MAX_ROUNDS <= 2
print('Retention rule:', retention_rule or 'Not yet supplied')


## 3. Implement the control loop

Open `student_work/week02.py` in the repository root, complete `reflect(session, initial_answer, max_rounds=2)`, and save it. The supplied session handles requests and recording; you decide what runs next.

`session.review(answer)` returns `decision` and `feedback`. For `revise`, request `session.revise(answer, review)` when permitted. Stop with `review_complete`, `needs_information`, `unchanged`, or `revision_limit`, as defined in the lab. Return `session.finish(answer, reason)`. Accept integer budgets from zero to two; zero is a boundary test, not the assessed comparison.

Add control tests in `student_work/test_week02.py` for early acceptance, missing information, an unchanged revision, the limit, and invalid budgets. Use scripted responses for these tests.

In [ ]:
import importlib
import student_work.week02 as student
student = importlib.reload(student)
print('Student implementation:', student.__file__)
print(Path(student.__file__).read_text(encoding='utf-8'))
if RUN_CONTROL_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', str(LAB_ROOT / 'student_work/test_week02.py'), '-q'], cwd=LAB_ROOT, check=True)
fixture_folder = ARTIFACT_ROOT / EXPERIMENT / 'fixture-check'
if RUN_STUDENT:
    fixture_records = run_comparison(student.reflect, task, ReviewFixture, fixture_folder, max_rounds=MAX_ROUNDS)
elif (fixture_folder / 'comparison.json').is_file():
    fixture_records = json.loads((fixture_folder / 'comparison.json').read_text(encoding='utf-8'))
else:
    fixture_records = []
    print('Student implementation has not been evaluated. Complete it, then enable RUN_STUDENT once.')
for record in fixture_records:
    print(record['variant'], record['stop_reason'], record['error'])


## 4. Run one pilot, then the ten samples

The **browser** route needs no API purchase. Python selects each request. Send the displayed prompt in a fresh chat with the same model and settings, then paste the complete reply into the capture panel. The program decides whether to continue. Save provider errors through the failure button. Human transfer is part of this route; it is not unattended execution.

Record the model label and settings actually shown. Use `None` only for unavailable information. The **adapter** route uses the configured course CLI/API backend. Never put credentials in the notebook. An approved Gemini adapter in Colab can read `GEMINI_API_KEY` from Colab Secrets; the browser route uses your existing chat account.

Each sample uses at most `1 + 4 * MAX_ROUNDS` logical requests across all paths. At two rounds, ten samples use at most 90 requests, including the retained pilot. Browser elapsed time includes transfer and waiting.

Enable `RUN_LIVE_PILOT` once after your fixture check succeeds. Inspect it, turn that flag off, then enable `RUN_LIVE_BATCH` once. Existing results are loaded rather than regenerated. Keep completed comparisons containing failures. An interrupted folder needs separate diagnosis; do not overwrite it.

In [ ]:
LIVE_ROUTE = 'browser'
LIVE_PROVIDER = 'Claude'  # The provider you actually use.
LIVE_MODEL = None  # Enter the displayed label, or None if it is not shown.
VISIBLE_SETTINGS = {'reasoning_mode': None, 'web_search': None, 'memory': None, 'custom_instructions': None}
group = ARTIFACT_ROOT / EXPERIMENT / 'live'
tasks = development_tasks()
print([(t['sample'], t['complaint']) for t in tasks])
print('Maximum logical requests:', len(tasks) * (1 + 4 * MAX_ROUNDS))
def live_factory():
    if LIVE_ROUTE == 'browser':
        from maintenance_course.review_browser import BrowserTextModel
        return BrowserTextModel(LIVE_PROVIDER, LIVE_MODEL, VISIBLE_SETTINGS)
    if LIVE_ROUTE != 'adapter':
        raise ValueError('Choose browser or adapter')
    if os.getenv('COURSE_BACKEND', 'gemini') == 'gemini' and not os.getenv('GEMINI_API_KEY'):
        from google.colab import userdata
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    from maintenance_course.models import make_model
    return make_model('live')
def run_or_load(current):
    folder = group / current['sample']
    saved = folder / 'comparison.json'
    if saved.is_file():
        records = json.loads(saved.read_text(encoding='utf-8'))
        expected_hash = __import__('hashlib').sha256(current['prompt'].encode()).hexdigest()
        assert all(r['task_hash'] == expected_hash for r in records), 'Inputs changed; start a separate comparison.'
        return records
    return run_comparison(student.reflect, current, live_factory, folder, max_rounds=MAX_ROUNDS)
if RUN_LIVE_PILOT or RUN_LIVE_BATCH:
    assert retention_rule.strip(), 'Write your retention rule before live requests.'
    assert fixture_records and not any(r['error'] for r in fixture_records), 'Complete and inspect the fixture check first.'
    group.mkdir(parents=True, exist_ok=True)
    plan = {'retention_rule': retention_rule, 'max_rounds': MAX_ROUNDS, 'route': LIVE_ROUTE, 'provider': LIVE_PROVIDER if LIVE_ROUTE == 'browser' else None, 'model': LIVE_MODEL if LIVE_ROUTE == 'browser' else os.getenv('COURSE_MODEL', os.getenv('GEMINI_MODEL')), 'settings': VISIBLE_SETTINGS if LIVE_ROUTE == 'browser' else {k: os.getenv(k) for k in ('COURSE_BACKEND', 'COURSE_EFFORT')}, 'samples': [t['sample'] for t in tasks]}
    plan_path = group / 'plan.json'
    if plan_path.exists():
        assert json.loads(plan_path.read_text()) == plan, 'Plan changed; start a separate comparison.'
    else:
        plan_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')
    if RUN_LIVE_PILOT:
        pilot = run_or_load(tasks[0])
        print(json.dumps(resource_rows(pilot), indent=2))
    if RUN_LIVE_BATCH:
        assert (group / tasks[0]['sample'] / 'comparison.json').is_file(), 'Inspect the pilot before the full set.'
        for current in tasks:
            print(current['sample'], json.dumps(resource_rows(run_or_load(current)), indent=2))
else:
    print('Live requests are disabled. Saved records remain available for analysis.')


## 5. Assess the answers and compare outcomes

The supplied scoring helper reads `ratings.json` from the comparison folder. To use that helper, complete the file through the file browser. Rate the initial answer and each path's selected final answer. Set `supported_facts`, `follows_rules`, and `completes_task` to JSON `true` or `false`, and add an evidence note. A request failure is an unsuccessful workflow outcome; keep it and explain it.

Judge against the task and sources, not the reviewer's approval. Ratings never enter model requests. The export counts corrected, regressed, unchanged success, and unchanged failure for each review path against the shared initial answer. Each path's denominator is ten.

Inspect at least one case closely. If no regression or correction failure occurs, examine an unchanged case and explain what this sample cannot establish.

In [ ]:
records = []
for current in tasks:
    saved = group / current['sample'] / 'comparison.json'
    if saved.is_file():
        records.extend(json.loads(saved.read_text(encoding='utf-8')))
ratings_path = group / 'ratings.json'
if len(records) == 30:
    conditions = {(r['mode'], r['backend'], r['model'], json.dumps(r['settings'], sort_keys=True), r['transport']) for r in records}
    assert len(conditions) == 1, 'Model, settings, mode or route changed; keep separate groups.'
    resolved = {tuple(r['resolved_models']) for r in records if r.get('resolved_models')}
    assert len(resolved) <= 1, 'Provider-reported model identities changed; retain and diagnose the group.'
    if not ratings_path.exists():
        ratings_path.write_text(json.dumps(rating_form(records), indent=2), encoding='utf-8')
    print('Complete this assessment file:', ratings_path)
else:
    print('Completed path records:', len(records), 'of 30. No quality assessment has been completed.')


## 6. Save the evidence and write your decision

To use the supplied export helper after completing the ratings, enable `EXPORT` once. It creates a folder under `submissions/week-02` containing requests, plan, ratings, outcomes, and resources. These output names and formats are defaults; you may organize the same evidence in notebook cells or separate files. Existing exports are preserved. Token counts remain unknown when unavailable. Shared generation is counted once; branch rows report additional requests only.

Explain your retention rule, paired outcomes, case analysis, resource tradeoffs, and decision. Include your research note and AI-use disclosure, and link or include the discussion and peer response. No positive result or fixed analysis word count is required.

Preserve your implementation, tests, and evidence in the evaluated commit. To download the helper's export as a ZIP, enable `DOWNLOAD`. Download the executed notebook through its File menu. Turn run/export/download flags off before the final clean-kernel execution.

Commit your executed notebook, implementation, tests, and evidence to your GitHub course repository and open a Week 2 pull request. Submit the pull-request URL and evaluated commit hash in Canvas using text entry. No separate notebook upload to Canvas is required. Give the instructor repository access.

In [ ]:
submission = LAB_ROOT / 'submissions/week-02' / EXPERIMENT
if EXPORT:
    assert len(records) == 30, 'Complete the ten-sample comparison first.'
    ratings = json.loads(ratings_path.read_text(encoding='utf-8'))
    assessment = export_comparison(submission, records, ratings)
    (submission / 'plan.json').write_bytes((group / 'plan.json').read_bytes())
    print(json.dumps(assessment['counts'], indent=2))
if (submission / 'transitions.json').is_file():
    print((submission / 'transitions.json').read_text(encoding='utf-8'))
else:
    print('No completed assessment export yet.')
if DOWNLOAD:
    assert submission.is_dir(), 'Complete the evidence export first.'
    archive_path = ARTIFACT_ROOT / (EXPERIMENT + '-evidence.zip')
    with zipfile.ZipFile(archive_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in (submission, group):
            for item in folder.rglob('*'):
                if item.is_file():
                    prefix = 'submissions/week-02/' + EXPERIMENT if folder == submission else 'request-records'
                    archive.write(item, str(PurePosixPath(prefix) / item.relative_to(folder).as_posix()))
        for name in ('week02.py', 'test_week02.py'):
            item = LAB_ROOT / 'student_work' / name
            if item.is_file():
                archive.write(item, 'student_work/' + name)
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        print('Evidence ZIP:', archive_path)
